In [0]:
import requests
import json
import uuid
import random
from copy import deepcopy
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType

BASE_URL = "https://dummyjson.com"
TARGET_TABLE = "retail_lakehouse_dr.bronze.api_snapshots"
RUN_ID = str(uuid.uuid4())
EXTRACTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Run ID: {RUN_ID}")
print(f"Extraction timestamp: {EXTRACTED_AT}")

Run ID: dd7e25a8-5b13-467f-b0ff-7bead2909794
Extraction timestamp: 2026-10-25T07:04:17.187788+00:00


In [0]:
def fetch_all(endpoint, entity_name):
    url = f"{BASE_URL}/{endpoint}"
    limit = 100
    skip = 0
    total = None
    records = []

    while total is None or skip < total:
        response = requests.get(url, params={"limit": limit, "skip": skip}, timeout=30)
        response.raise_for_status()
        result = response.json()
        batch = result.get(entity_name, [])

        if total is None:
            total = result["total"]
        if not batch:
            break

        records.extend(batch)
        skip += len(batch)

    print(f"{endpoint}: {len(records)} source records extracted")
    return records


products = fetch_all("products", "products")
carts = fetch_all("carts", "carts")

bronze_schema = StructType([
    StructField("source", StringType(), False),
    StructField("record_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("extracted_at", StringType(), False),
    StructField("payload_json", StringType(), False)
])

def make_bronze_row(source, record_id, payload):
    return (source, str(record_id), RUN_ID, EXTRACTED_AT, json.dumps(payload, ensure_ascii=False, separators=(",", ":")))

products: 194 source records extracted
carts: 208 source records extracted


In [0]:
# Raw State
bronze_rows = []
for product in products:
    bronze_rows.append(make_bronze_row("products", product["id"], product))
for cart in carts:
    bronze_rows.append(make_bronze_row("carts", cart["id"], cart))
print(f"Raw API snapshot rows prepared: {len(bronze_rows)}")


Raw API snapshot rows prepared: 402


In [0]:
# Read the latest simulated product state from previous Job runs.
# On the first run, there are no simulated rows and state is initialized from the API.

previous_state = {}

previous_rows = spark.sql(f"""
    WITH ranked AS (
        SELECT
            record_id,
            payload_json,
            ROW_NUMBER() OVER (
                PARTITION BY record_id
                ORDER BY extracted_at DESC, run_id DESC
            ) AS row_num
        FROM {TARGET_TABLE}
        WHERE source = 'products_simulated'
    )
    SELECT record_id, payload_json
    FROM ranked
    WHERE row_num = 1
""").collect()

for row in previous_rows:
    previous_state[str(row["record_id"])] = json.loads(row["payload_json"])

print(f"Previous simulated product states loaded: {len(previous_state)}")


Previous simulated product states loaded: 194


In [0]:
# If there's no previous step it fills with the first load, if there's a previous step it alters the data (simulated)...

simulated_products = []

for source_product in products:
    product = deepcopy(source_product)
    product_id = str(product["id"])
    api_price = float(source_product["price"])
    api_stock = int(source_product.get("stock", 0))
    previous = previous_state.get(product_id)

    if previous is None:
        previous_simulated_price = api_price
        previous_simulated_stock = api_stock
        simulated_price = round(max(0.01, api_price * (1 + random.uniform(-0.02, 0.02))), 2)
        simulated_stock = max(0, api_stock + random.randint(-5, 8))
        change_type = "INITIALIZED"
    else:
        metadata = previous.get("_simulation", {})
        previous_api_price = float(metadata.get("api_price", previous.get("price", api_price)))
        previous_api_stock = int(metadata.get("api_stock", previous.get("stock", api_stock)))
        previous_simulated_price = float(metadata.get("simulated_price", previous.get("price", api_price)))
        previous_simulated_stock = int(metadata.get("simulated_stock", previous.get("stock", api_stock)))

        # Use the first value, if not, introduce a small simulated price/stock movement.
        source_price_ratio = api_price / previous_api_price if previous_api_price > 0 else 1.0
        simulated_price = round(max(0.01, previous_simulated_price * source_price_ratio * (1 + random.uniform(-0.02, 0.02))), 2)
        simulated_stock = max(0, previous_simulated_stock + (api_stock - previous_api_stock) + random.randint(-5, 8))
        change_type = "UPDATED"

    price_delta = round(simulated_price - previous_simulated_price, 2)
    stock_delta = simulated_stock - previous_simulated_stock

    # This modified copy is explicitly labeled as simulated. Raw API rows remain unchanged.
    product["price"] = simulated_price
    product["stock"] = simulated_stock
    product["_simulation"] = {
        "run_id": RUN_ID,
        "updated_at": EXTRACTED_AT,
        "origin": "dummyjson_plus_simulated_variation",
        "api_price": api_price,
        "api_stock": api_stock,
        "simulated_price": simulated_price,
        "simulated_stock": simulated_stock,
        "price_delta": price_delta,
        "stock_delta": stock_delta,
        "change_type": change_type
    }

    simulated_products.append(product)
    bronze_rows.append(make_bronze_row("products_simulated", product_id, product))

print(f"Simulated product snapshots prepared: {len(simulated_products)}")

bronze_df = spark.createDataFrame(bronze_rows, schema=bronze_schema)
print(f"Total Bronze rows to append this run: {bronze_df.count()}")
display(bronze_df.groupBy("source").count().orderBy("source"))

Simulated product snapshots prepared: 194


In [0]:
# Generate new synthetic carts on each run, using the data from products.

# synthetic_carts = []
# product_pool = simulated_products

# for cart_number in range(SYNTHETIC_CARTS_PER_RUN):
#     selected_count = random.randint(1, min(5, len(product_pool)))
#     selected_products = random.sample(product_pool, selected_count)

#     cart_products = []
#     total = 0.0
#     discounted_total = 0.0
#     total_quantity = 0

#     for product in selected_products:
#         quantity = random.randint(1, 4)
#         price = float(product["price"])
#         discount = float(product.get("discountPercentage", 0))
#         line_total = round(price * quantity, 2)
#         line_discounted_total = round(price * (1 - discount / 100) * quantity, 2)

#         cart_products.append({
#             "productId": int(product["id"]),
#             "title": product["title"],
#             "price": price,
#             "quantity": quantity,
#             "total": line_total,
#             "discountPercentage": discount,
#             "discountedTotal": line_discounted_total
#         })

#         total += line_total
#         discounted_total += line_discounted_total
#         total_quantity += quantity

#     cart_id = f"sim_{RUN_ID[:8]}_{cart_number + 1}"
#     synthetic_cart = {
#         "id": cart_id,
#         "userId": random.randint(1, 208),
#         "products": cart_products,
#         "total": round(total, 2),
#         "discountedTotal": round(discounted_total, 2),
#         "totalProducts": len(cart_products),
#         "totalQuantity": total_quantity,
#         "_simulation": {
#             "run_id": RUN_ID,
#             "created_at": EXTRACTED_AT,
#             "origin": "synthetic_cart_generator"
#         }
#     }

#     synthetic_carts.append(synthetic_cart)
#     bronze_rows.append(make_bronze_row("carts_simulated", cart_id, synthetic_cart))

# print(f"Synthetic carts generated: {len(synthetic_carts)}")

Synthetic carts generated: 5
Total Bronze rows to append this run: 601


source,count
carts,208
carts_simulated,5
products,194
products_simulated,194


In [0]:
# Append raw snapshots and simulated records to the same Bronze Delta table.
(bronze_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE))
print(f"Successfully appended this run to {TARGET_TABLE}")

Successfully appended this run to retail_lakehouse_dr.bronze.api_snapshots


total_snapshots,unique_products,total_runs,product_versions
18236,194,93,18042


record_id,title,simulated_price,simulated_stock,price_delta,stock_delta,run_id,extracted_at
1,Essence Mascara Lash Princess,11.96,229,0.09,6,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
10,Gucci Bloom Eau de,80.45,229,0.24,-4,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
100,Apple Airpods,110.93,234,1.25,0,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
101,Apple AirPods Max Silver,509.42,236,8.56,-4,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
102,Apple Airpower Wireless Charger,78.28,203,1.01,3,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
103,Apple HomePod Mini Cosmic Grey,94.92,162,-1.35,-2,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
104,Apple iPhone Charger,18.12,127,-0.24,-5,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
105,Apple MagSafe Battery Pack,103.27,137,0.38,-1,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
106,Apple Watch Series 4 Gold,398.83,149,-4.72,0,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00
107,Beats Flex Wireless Earphones,49.66,209,0.68,5,dd7e25a8-5b13-467f-b0ff-7bead2909794,2026-10-25T07:04:17.187788+00:00


In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS total_snapshots,
        COUNT(DISTINCT record_id) AS unique_products,
        COUNT(DISTINCT run_id) AS total_runs,
        COUNT(DISTINCT CONCAT(
            record_id, '_', run_id
        )) AS product_versions
    FROM retail_lakehouse_dr.bronze.api_snapshots
    WHERE source = 'products_simulated'
"""))


In [0]:
display(spark.sql(f"""
    SELECT
        record_id,
        get_json_object(payload_json, '$.title') AS title,
        get_json_object(payload_json, '$.price') AS simulated_price,
        get_json_object(payload_json, '$.stock') AS simulated_stock,
        get_json_object(payload_json, '$._simulation.price_delta') AS price_delta,
        get_json_object(payload_json, '$._simulation.stock_delta') AS stock_delta,
        run_id,
        extracted_at
    FROM {TARGET_TABLE}
    WHERE source = 'products_simulated'
    ORDER BY extracted_at DESC, record_id
    LIMIT 20
"""))

In [0]:
display(spark.sql("""
    SELECT
        source,
        COUNT(*) AS total_records,
        COUNT(DISTINCT run_id) AS total_runs,
        COUNT(DISTINCT record_id) AS distinct_record_ids,
        MAX(extracted_at) AS latest_extraction
    FROM retail_lakehouse_dr.bronze.api_snapshots
    GROUP BY source
    ORDER BY source
"""))

source,total_records,total_runs,distinct_record_ids,latest_extraction
carts,20176,96,208,2026-10-25T07:04:17.187788+00:00
carts_simulated,470,93,465,2026-10-25T07:04:17.187788+00:00
products,18818,96,194,2026-10-25T07:04:17.187788+00:00
products_simulated,18236,93,194,2026-10-25T07:04:17.187788+00:00


In [0]:
display(spark.sql("""
    SELECT
        record_id AS product_id,
        get_json_object(payload_json, '$.title') AS title,
        COUNT(*) AS snapshots,
        COUNT(DISTINCT get_json_object(
            payload_json, '$.price'
        )) AS distinct_prices,
        COUNT(DISTINCT get_json_object(
            payload_json, '$.stock'
        )) AS distinct_stock_values,
        MIN(CAST(get_json_object(
            payload_json, '$.price'
        ) AS DOUBLE)) AS min_price,
        MAX(CAST(get_json_object(
            payload_json, '$.price'
        ) AS DOUBLE)) AS max_price,
        MIN(CAST(get_json_object(
            payload_json, '$.stock'
        ) AS INT)) AS min_stock,
        MAX(CAST(get_json_object(
            payload_json, '$.stock'
        ) AS INT)) AS max_stock
    FROM retail_lakehouse_dr.bronze.api_snapshots
    WHERE source = 'products_simulated'
    GROUP BY record_id, get_json_object(payload_json, '$.title')
    ORDER BY record_id
    LIMIT 30
"""))

product_id,title,snapshots,distinct_prices,distinct_stock_values,min_price,max_price,min_stock,max_stock
1,Essence Mascara Lash Princess,94,81,63,9.77,11.96,85,229
10,Gucci Bloom Eau de,94,90,58,71.85,85.08,97,233
100,Apple Airpods,94,91,65,106.61,132.03,69,234
101,Apple AirPods Max Silver,94,93,69,471.86,567.84,51,240
102,Apple Airpower Wireless Charger,94,91,76,76.55,86.38,0,203
103,Apple HomePod Mini Cosmic Grey,94,90,64,89.95,106.05,19,164
104,Apple iPhone Charger,94,80,60,17.07,20.76,22,132
105,Apple MagSafe Battery Pack,94,88,66,90.52,107.06,0,144
106,Apple Watch Series 4 Gold,94,93,54,354.4,407.31,36,149
107,Beats Flex Wireless Earphones,94,89,65,45.02,52.05,45,209


In [0]:
display(spark.sql("""
    SELECT
        record_id AS product_id,
        get_json_object(payload_json, '$.price') AS price,
        get_json_object(payload_json, '$.stock') AS stock,
        get_json_object(
            payload_json, '$._simulation.price_delta'
        ) AS price_delta,
        get_json_object(
            payload_json, '$._simulation.stock_delta'
        ) AS stock_delta,
        run_id,
        extracted_at
    FROM retail_lakehouse_dr.bronze.api_snapshots
    WHERE source = 'products_simulated'
      AND record_id = '1'
    ORDER BY extracted_at
"""))

product_id,price,stock,price_delta,stock_delta,run_id,extracted_at
1,10.17,94,0.18,-5,4e138dad-0a34-4b1a-8505-354ed9683464,2026-10-19T04:47:33.544437+00:00
1,9.97,100,-0.2,6,a64460da-02e3-460f-85e0-ba0073b36608,2026-10-19T04:52:15.241231+00:00
1,9.8,96,-0.17,-4,a28f5747-55df-4a46-b1c7-ec2cdc76165b,2026-10-19T04:52:48.002025+00:00
1,9.8,96,-0.17,-4,a28f5747-55df-4a46-b1c7-ec2cdc76165b,2026-10-19T04:52:48.002025+00:00
1,9.85,93,0.05,-3,cc91c4ae-dd13-40e4-afef-cf0f82dd43f2,2026-10-19T04:58:34.702458+00:00
1,9.82,93,-0.03,0,e8863497-b708-4cf2-842a-3f900c7d7817,2026-10-19T05:10:45.408492+00:00
1,9.84,89,0.02,-4,d964b8c2-4d3f-45dc-b9e8-58e1666cb914,2026-10-19T06:10:46.948965+00:00
1,9.96,85,0.12,-4,49694bd4-33f9-403b-a3ad-fddfe2e48781,2026-10-19T07:10:48.414527+00:00
1,9.77,88,-0.19,3,1bf8e2ca-9a7b-499b-aef3-0da06ee56289,2026-10-19T08:10:48.746049+00:00
1,9.92,91,0.15,3,7012177e-9a43-41b8-8bb0-cf6a7095bc20,2026-10-19T09:10:49.234984+00:00
